<a href="https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/milestone_1_csv_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 1 — CSV data audit and current-label prototype

This is the fast development path and follows the EDA notebook's `data/raw/content_refresh_anonymized.csv` convention. It uses the 30,000-row starter dataset and does not connect to DuckDB or Hugging Face. Use it for the data audit, feature/missingness decisions, baseline and model prototyping.

**Scope:** identify or prioritize pages labelled as currently declining by the supplied `trend_direction` field. The CSV is a snapshot, so it cannot establish that predictors were available before the label window. Do not describe these results as future early warnings. A later full-release run can verify the dated window contract and generalization.

In [1]:
# Match the Colab/local repo setup pattern used in the EDA notebook, but avoid
# installing the full requirements set for this small CSV-only stage.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/huseyinTozluyurt/Flyrank-Internship-MachineLearning'
REPO_DIR = 'Flyrank-Internship-MachineLearning'
CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')

if IN_COLAB and not CSV_PATH.exists():
    if not Path(REPO_DIR).is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
else:
    while not CSV_PATH.exists() and Path.cwd() != Path('/'):
        os.chdir('..')
        CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')

if not CSV_PATH.exists():
    raise FileNotFoundError('Starter CSV not found at data/raw/content_refresh_anonymized.csv; open this notebook from the FlyRank repo or clone it in Colab.')
print('CSV:', CSV_PATH.resolve())

CSV: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [2]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv(CSV_PATH)
required = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct',
    'days_since_last_update', 'content_age_days', 'impressions_90d',
    'avg_position', 'ctr', 'content_type'
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f'Starter CSV does not match the expected data contract. Missing: {missing}')

PAGE_ID, GROUP_ID = 'content_id', 'client_id'
TARGET_SOURCE, TARGET = 'trend_direction', 'is_declining_label'
df[TARGET_SOURCE] = df[TARGET_SOURCE].astype('string').str.strip().str.lower()
df[TARGET] = df[TARGET_SOURCE].map({'down': 1, 'new': 0, 'up': 0, 'flat': 0, 'stable': 0}).astype('Int64')

for col in ['days_since_last_update', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Rows: {len(df):,}; columns: {len(df.columns)}')
print(f'Unique content IDs: {df[PAGE_ID].nunique(dropna=True):,}; clients: {df[GROUP_ID].nunique(dropna=True):,}')
print('Trend labels, including missing/unrecognized values:')
print(df[TARGET_SOURCE].value_counts(dropna=False))
print('Binary target counts:')
print(df[TARGET].value_counts(dropna=False).sort_index())

Rows: 30,000; columns: 45
Unique content IDs: 30,000; clients: 32
Trend labels, including missing/unrecognized values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: Int64
Binary target counts:
is_declining_label
0    13738
1    16262
Name: count, dtype: Int64


## Data and feature contract

- **Unit:** one unique `content_id` within `client_id`; retain both IDs for auditing and grouped client holdout, never use either as a predictor.
- **Target:** `1` for `trend_direction == 'down'`; `0` for known `new`, `up`, `flat`, or `stable` labels. Unknown values stay unlabeled.
- **Prototype features matching the prior model notebooks:** `days_since_last_update`, `content_age_days`, `impressions_90d`, `avg_position`, `ctr`, plus `staleness_pos_interaction`. Exclude `trend_pct`, `trend_direction`, the target, and IDs.
- **Temporal limitation:** this snapshot does not prove that its 90-day GSC aggregates precede the trend label, which is derived from 30-day comparisons. This is current-label classification only; the warehouse is required to establish an earlier feature window.
- **Missingness:** `avg_position == 0` means no position data; keep it missing and add an indicator. Keep actual zero impressions distinct from unavailable GSC. Do not fill every null with zero. Fit any imputer on training folds only and include missingness indicators, especially for `word_count` and other content-type-dependent fields.

In [5]:
# Audit page grain and missingness without silently dropping rows.
page_unique = ~df[PAGE_ID].duplicated(keep=False)
ids_valid = df[PAGE_ID].notna() & df[GROUP_ID].notna()
label_valid = df[TARGET].notna()

# In this starter snapshot, positive impressions indicate GSC volume is present.
# avg_position == 0 is the documented no-position sentinel, not a valid rank.
df['gsc_impressions_present'] = df['impressions_90d'].notna()
df['gsc_position_present'] = df['avg_position'].notna() & df['avg_position'].ne(0)
df['gsc_ctr_present'] = df['ctr'].notna()
df['gsc_core_available'] = df['gsc_impressions_present']

# No time-provenance gate is applied to CSV prototype eligibility. This only
# determines whether the row can participate in current-label code/model work.
df['eligible_for_csv_prototype'] = ids_valid & page_unique & label_valid

def exclusion_reason(i):
    reasons=[]
    if not ids_valid.loc[i]: reasons.append('missing_page_or_client_id')
    if not page_unique.loc[i]: reasons.append('duplicate_content_id')
    if not label_valid.loc[i]: reasons.append('invalid_or_missing_target')
    return ';'.join(reasons)
df['exclusion_reasons'] = [exclusion_reason(i) for i in df.index]

prototype_features = [
    'days_since_last_update', 'content_age_days', 'impressions_90d',
    'avg_position', 'ctr'
]
df['staleness_pos_interaction'] = df['days_since_last_update'] * df['avg_position'].where(df['avg_position'].ne(0))
prototype_features.append('staleness_pos_interaction')

missingness = pd.DataFrame({
    'field': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'missing_rate': [float(df[c].isna().mean()) for c in df.columns],
    'distinct_count': [int(df[c].nunique(dropna=True)) for c in df.columns],
})
feature_missingness_by_content_type = (
    df.groupby('content_type', dropna=False)[prototype_features]
      .agg(lambda col: col.isna().mean())
      .reset_index()
)
client_availability = (
    df.groupby(GROUP_ID, dropna=False)
      .agg(page_rows=(PAGE_ID,'size'), eligible_pages=('eligible_for_csv_prototype','sum'),
           impressions_missing_rate=('gsc_impressions_present',lambda s: 1-s.mean()),
           position_no_data_rate=('gsc_position_present',lambda s: 1-s.mean()),
           ctr_missing_rate=('gsc_ctr_present',lambda s: 1-s.mean()))
      .reset_index()
)

print(f'Duplicate content ID rows: {int((~page_unique).sum()):,}')
print(f'Eligible CSV prototype rows: {int(df["eligible_for_csv_prototype"].sum()):,}')
display(missingness.sort_values('missing_rate',ascending=False).head(20))


Duplicate content ID rows: 0
Eligible CSV prototype rows: 30,000


,field,dtype,missing_count,missing_rate,distinct_count
10,provider_used,object,21438,0.714600,2
8,word_count,float64,7699,0.256633,5476
9,char_count,float64,7699,0.256633,14839
33,word_count_tier,object,7699,0.256633,4
34,char_count_tier,object,7699,0.256633,4
11,model_used,object,5733,0.191100,5
43,trend_pct,float64,3388,0.112933,2712
4,competition_level,object,2610,0.087000,3
2,search_volume,float64,2468,0.082267,41
5,cpc,float64,2468,0.082267,915


In [6]:
display(feature_missingness_by_content_type)

,content_type,days_since_last_update,content_age_days,impressions_90d,avg_position,ctr,staleness_pos_interaction
0,comparison article,0.0,0.0,0.0,0.0,0.0,0.000000
1,feedly article,0.0,0.0,0.0,0.0,0.0,0.348282
2,keyword article,0.0,0.0,0.0,0.0,0.0,0.017459


In [7]:
OUTPUT_DIR = Path('work/outputs/milestone_1_csv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    'source': str(CSV_PATH),
    'rows': int(len(df)),
    'unique_content_ids': int(df[PAGE_ID].nunique(dropna=True)),
    'unique_clients': int(df[GROUP_ID].nunique(dropna=True)),
    'duplicate_content_id_rows': int((~page_unique).sum()),
    'eligible_csv_prototype_rows': int(df['eligible_for_csv_prototype'].sum()),
    'target_counts': {str(k): int(v) for k,v in df[TARGET].value_counts(dropna=False).items()},
    'prototype_features': prototype_features,
    'temporal_provenance': 'unverified in CSV snapshot; current-label prototype only',
}
contract = {
    'unit': 'one unique content_id within client_id',
    'target': TARGET,
    'target_rule': '1 iff trend_direction is down; 0 for known new/up/flat/stable; otherwise unlabeled',
    'prototype_features': prototype_features,
    'excluded_predictors': ['trend_pct','trend_direction',TARGET,PAGE_ID,GROUP_ID],
    'gsc_policy': 'avg_position zero is no-data; do not zero-impute missing GSC fields; add missingness indicators',
    'missing_data_policy': 'fit imputers inside training folds; report missingness by content_type',
    'split_policy': 'group holdout by client_id for prototype evaluation',
    'claim_scope': 'current-label detection only; no future early-warning claim from this snapshot',
    'full_release_required_for': 'verify feature and label window separation and final warehouse performance',
}

missingness.to_csv(OUTPUT_DIR/'field_missingness.csv',index=False)
feature_missingness_by_content_type.to_csv(OUTPUT_DIR/'feature_missingness_by_content_type.csv',index=False)
client_availability.to_csv(OUTPUT_DIR/'gsc_availability_by_client.csv',index=False)
df.to_csv(OUTPUT_DIR/'page_level_audit.csv',index=False)
df.loc[df['eligible_for_csv_prototype'],[PAGE_ID,GROUP_ID,TARGET,*prototype_features]].to_csv(OUTPUT_DIR/'csv_prototype_dataset.csv',index=False)
(OUTPUT_DIR/'audit_summary.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
(OUTPUT_DIR/'data_contract.json').write_text(json.dumps(contract,indent=2)+'\n',encoding='utf-8')
print(json.dumps(summary,indent=2))
print('CSV audit outputs saved to:',OUTPUT_DIR.resolve())

{
  "source": "data/raw/content_refresh_anonymized.csv",
  "rows": 30000,
  "unique_content_ids": 30000,
  "unique_clients": 32,
  "duplicate_content_id_rows": 0,
  "eligible_csv_prototype_rows": 30000,
  "target_counts": {
    "1": 16262,
    "0": 13738
  },
  "prototype_features": [
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "avg_position",
    "ctr",
    "staleness_pos_interaction"
  ],
  "temporal_provenance": "unverified in CSV snapshot; current-label prototype only"
}
CSV audit outputs saved to: /content/flyrank-ml-internship-starter/work/outputs/milestone_1_csv
